<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS05_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS05 — Clustering Berbasis Kepadatan (HDBSCAN) pada Data Properti

Jobsheet asli memakai HDBSCAN pada dataset digit. HDBSCAN adalah pengembangan DBSCAN
yang **tidak perlu menentukan jumlah klaster** dan dapat menandai titik yang tidak
termasuk klaster mana pun sebagai **noise** (label `-1`).

Versi PBL ini menjalankan HDBSCAN pada atribut fisik properti, lalu membandingkannya
dengan K-Means dari JS04.

## Langkah 1 — Muat Data dan Siapkan Fitur

Fitur dan aturan pembersihan sama dengan JS04 agar kedua hasil clustering sebanding.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, normalized_mutual_info_score)

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR] = df[FITUR].fillna(df[FITUR].median())

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

X = StandardScaler().fit_transform(df[FITUR])
print(f'Data siap klaster: {X.shape}')

Data siap klaster: (56170, 4)


## Langkah 2 — Menyetel `min_cluster_size` dan `min_samples`

| Parameter | Arti |
|---|---|
| `min_cluster_size` | jumlah minimum titik agar sebuah kelompok dianggap klaster |
| `min_samples` | jumlah tetangga minimum untuk menentukan kepadatan lokal; naik → lebih banyak noise |

Penyetelan dilakukan pada sampel 10.000 baris agar cepat. Kriteria pemilihan:
**silhouette tertinggi** pada titik non-noise, dengan **proporsi noise wajar** (di bawah
±30%).

In [2]:
SAMPEL = df.sample(10_000, random_state=RANDOM_STATE)
X_sampel = StandardScaler().fit_transform(SAMPEL[FITUR])

hasil_tuning = []
for mcs in [100, 200, 500, 1000]:
    for ms in [10, 20, 50]:
        hdb = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit(X_sampel)
        label = hdb.labels_
        n_klaster = len(set(label)) - (1 if -1 in label else 0)
        noise = (label == -1).mean()

        if n_klaster >= 2:
            mask = label != -1
            sil = silhouette_score(X_sampel[mask], label[mask],
                                   sample_size=3000, random_state=RANDOM_STATE)
        else:
            sil = np.nan

        hasil_tuning.append({'min_cluster_size': mcs, 'min_samples': ms,
                             'n_klaster': n_klaster, 'noise_%': round(noise * 100, 1),
                             'silhouette': sil})

tuning = pd.DataFrame(hasil_tuning).sort_values('silhouette', ascending=False)
tuning.round(4)

,min_cluster_size,min_samples,n_klaster,noise_%,silhouette
5,200,50,8,18.6,0.5706
4,200,20,8,15.6,0.5334
2,100,50,15,44.3,0.5125
11,1000,50,2,27.3,0.5074
8,500,50,4,20.1,0.4951
1,100,20,18,38.8,0.4831
3,200,10,8,11.3,0.4772
10,1000,20,3,15.1,0.4540
9,1000,10,4,14.1,0.4436
7,500,20,5,11.6,0.4372


## Langkah 3 — Latih HDBSCAN pada Seluruh Data

Konfigurasi terbaik dari tabel di atas dipakai pada 56.170 baris.

In [3]:
baris_terbaik = tuning.dropna().iloc[0]
MCS = int(baris_terbaik['min_cluster_size'])
MS = int(baris_terbaik['min_samples'])
print(f'Konfigurasi terpilih: min_cluster_size={MCS}, min_samples={MS}')

hdb = HDBSCAN(min_cluster_size=MCS, min_samples=MS).fit(X)
df['klaster_hdb'] = hdb.labels_

n_klaster = len(set(df['klaster_hdb'])) - (1 if -1 in df['klaster_hdb'].values else 0)
noise = (df['klaster_hdb'] == -1).mean()
print(f'Jumlah klaster : {n_klaster}')
print(f'Proporsi noise : {noise:.2%}')
print(f'Ukuran klaster : {pd.Series(df["klaster_hdb"]).value_counts().sort_index().to_dict()}')

Konfigurasi terpilih: min_cluster_size=200, min_samples=50


Jumlah klaster : 40
Proporsi noise : 30.13%
Ukuran klaster : {-1: 16926, 0: 646, 1: 256, 2: 2115, 3: 699, 4: 1754, 5: 922, 6: 1437, 7: 259, 8: 742, 9: 8154, 10: 1039, 11: 4557, 12: 328, 13: 264, 14: 488, 15: 218, 16: 2863, 17: 2220, 18: 201, 19: 212, 20: 341, 21: 390, 22: 461, 23: 386, 24: 305, 25: 712, 26: 749, 27: 362, 28: 766, 29: 557, 30: 264, 31: 307, 32: 230, 33: 418, 34: 256, 35: 600, 36: 644, 37: 335, 38: 1353, 39: 434}


## Langkah 4 — Profil Klaster

Klaster HDBSCAN diurutkan berdasarkan median harga agar mudah dibaca. Baris `-1`
adalah noise.

In [4]:
profil = (df.groupby('klaster_hdb')
            .agg(jumlah=('harga', 'size'),
                 luas_tanah=('luas_m2', 'mean'),
                 luas_bangunan=('luas_bangunan', 'mean'),
                 kamar=('jumlah_kamar', 'mean'),
                 median_harga=('harga', 'median'))
            .round(1)
            .sort_values('median_harga'))
profil['median_harga'] = profil['median_harga'].map(lambda v: f'Rp {v:,.0f}')
profil['persen'] = (profil['jumlah'] / len(df) * 100).round(2)
profil

,jumlah,luas_tanah,luas_bangunan,kamar,median_harga,persen
klaster_hdb,,,,,,
28,766,61.1,29.9,2.0,"Rp 265,171,176",1.36
23,386,92.2,30.3,2.0,"Rp 279,874,000",0.69
26,749,75.7,29.6,2.0,"Rp 280,000,000",1.33
38,1353,71.9,36.0,2.0,"Rp 290,000,000",2.41
34,256,65.3,36.0,2.0,"Rp 293,696,396",0.46
37,335,101.4,36.0,2.0,"Rp 299,900,000",0.60
35,600,60.0,35.9,2.0,"Rp 306,906,000",1.07
33,418,84.9,36.0,2.0,"Rp 307,909,460",0.74
39,434,78.3,36.0,2.0,"Rp 314,500,000",0.77


## Langkah 5 — Bandingkan dengan K-Means dan Kelas Harga

HDBSCAN tidak memaksa semua titik masuk klaster, jadi perbandingan dilakukan pada
**titik non-noise** saja. K-Means `k = 3` dilatih pada data yang sama.

In [5]:
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X)
df['klaster_km'] = km.labels_

mask = df['klaster_hdb'] != -1
X_nn = X[mask]
label_hdb = df.loc[mask, 'klaster_hdb']
label_km = df.loc[mask, 'klaster_km']

print('Perbandingan pada titik non-noise:')
print(f'  Silhouette HDBSCAN : {silhouette_score(X_nn, label_hdb, sample_size=5000, random_state=RANDOM_STATE):.4f}')
print(f'  Silhouette K-Means : {silhouette_score(X_nn, label_km, sample_size=5000, random_state=RANDOM_STATE):.4f}')
print(f'  Davies-Bouldin HDBSCAN : {davies_bouldin_score(X_nn, label_hdb):.4f}')
print(f'  Davies-Bouldin K-Means : {davies_bouldin_score(X_nn, label_km):.4f}')

print('\nKesesuaian HDBSCAN dengan kelas harga (non-noise):')
print(f'  ARI : {adjusted_rand_score(df.loc[mask, "kelas_harga"], label_hdb):.4f}')
print(f'  NMI : {normalized_mutual_info_score(df.loc[mask, "kelas_harga"], label_hdb):.4f}')
print('\nKesesuaian K-Means dengan kelas harga (non-noise):')
print(f'  ARI : {adjusted_rand_score(df.loc[mask, "kelas_harga"], label_km):.4f}')
print(f'  NMI : {normalized_mutual_info_score(df.loc[mask, "kelas_harga"], label_km):.4f}')

Perbandingan pada titik non-noise:


  Silhouette HDBSCAN : 0.4375


  Silhouette K-Means : 0.4667
  Davies-Bouldin HDBSCAN : 0.8838
  Davies-Bouldin K-Means : 0.7232

Kesesuaian HDBSCAN dengan kelas harga (non-noise):
  ARI : 0.0562
  NMI : 0.1666

Kesesuaian K-Means dengan kelas harga (non-noise):
  ARI : 0.1344
  NMI : 0.1904


## Langkah 6 — Visualisasi (PCA 2D)

Noise digambar terpisah agar terlihat di wilayah mana HDBSCAN menilai kepadatan terlalu
rendah.

In [6]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca.fit_transform(X)
df['pc1'], df['pc2'] = Z[:, 0], Z[:, 1]

viz = df.sample(6000, random_state=RANDOM_STATE)
viz['label_viz'] = np.where(viz['klaster_hdb'] == -1, 'noise',
                            'klaster ' + viz['klaster_hdb'].astype(str))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=viz, x='pc1', y='pc2', hue='label_viz', s=12, alpha=0.6,
                palette='tab10', ax=ax[0])
ax[0].set_title('HDBSCAN (PCA 2D)')

sns.scatterplot(data=viz, x='pc1', y='pc2', hue='klaster_km', palette='viridis',
                s=12, alpha=0.6, ax=ax[1])
ax[1].set_title('K-Means k=3 (PCA 2D)')

plt.tight_layout()
plt.show()

print(f'Varians PC1+PC2: {pca.explained_variance_ratio_.sum():.2%}')

Varians PC1+PC2: 86.77%


**Analisis JS05:**

- HDBSCAN terbaik pada `min_cluster_size = 200`, `min_samples = 50`, menghasilkan **40 klaster** dengan **noise 30,13%** (16.926 dari 56.170 baris). Jadi hasilnya bukan "satu klaster besar": justru banyak klaster kecil-kecil.
- Klaster yang terbentuk sangat spesifik, mis. kelompok rumah 60–100 m² berharga Rp265–360 juta, atau kelompok rumah besar berharga miliaran. Ini konsekuensi data tabular kontinu: HDBSCAN memecah data menjadi gugus kepadatan lokal, bukan segmen pasar yang rapi.
- Pada titik non-noise, **K-Means sedikit unggul**: silhouette 0,4667 vs 0,4375; Davies-Bouldin 0,7232 vs 0,8838. Namun perbandingan ini tidak sepenuhnya adil karena HDBSCAN membuang 30% titik tersulit.
- Kesesuaian dengan kelas harga juga rendah: ARI HDBSCAN **0,0562** dan NMI **0,1666**, lebih rendah dari K-Means pada subset yang sama (ARI 0,1344; NMI 0,1904). Artinya clustering fitur fisik **tidak menemukan** batas Murah/Sedang/Mahal.
- Kelebihan HDBSCAN di sini adalah kemampuannya menandai **properti outlier** (mis. luas sangat besar dengan kamar sedikit, atau data hasil scraping yang aneh). Daftar noise bisa dipakai untuk audit data, bukan untuk segmentasi pasar.
- Untuk segmentasi yang mudah ditafsirkan, K-Means dengan `k` manual (JS04) lebih praktis; HDBSCAN lebih cocok sebagai alat deteksi anomali.
